# AP01 - Fehleranalyse

## SQL-Fehleranalyse

### Die zehn Kernfragen
1. Fehlen Objekte ohne Treffer? → `INNER JOIN` vs. `LEFT JOIN` prüfen, bei LEFT JOIN muss die vollständig benötigte Objekttabelle links stehen
2. Sollte eine Gruppe 0 haben, zeigt aber 1? → Bei LEFT JOIN zählt `COUNT(*)` auch die ergänzte Zeile, mit `COUNT(rechte_id)` nur echte Treffer zählen und dafür eine ID verwenden, die bei echten Treffern niemals NULL ist
3. Gesucht sind «noch nie» oder «ohne»? → `LEFT JOIN ... WHERE rechte_id IS NULL` oder `NOT EXISTS` verwenden, für die NULL-Prüfung eine rechte ID wählen, die bei echten Treffern niemals NULL ist
4. Filter auf Einzelzeilen? → `WHERE` verwenden, Filter auf Gruppenkennzahlen mit `HAVING COUNT(...)`, `HAVING AVG(...)` usw. umsetzen
5. Aggregation «pro X»? → `GROUP BY X` verwenden, Personen nach ID statt nur nach Namen gruppieren und nicht aggregierte Ausgabespalten grundsätzlich mitgruppieren
6. Spalten aus mehreren Tabellen benötigt? → JOIN-Pfad über passende Primär- und Fremdschlüssel prüfen, jede Verbindung braucht eine passende ON-Bedingung, ausser beim beabsichtigten CROSS JOIN
7. Ungültige Kategorien gesucht? → `NOT IN ('gültig1', 'gültig2')` verwenden, Gross- und Kleinschreibung sowie Leerzeichen beachten und fehlende Kategorien bei Bedarf mit `OR spalte IS NULL` erfassen
8. NULL geprüft? → `IS NULL` oder `IS NOT NULL` verwenden, niemals `= NULL`, leerer Text `''` ist kein NULL und auch `<>`, `NOT IN` und `NOT LIKE` wählen NULL nicht automatisch aus
9. Sortierung gefordert? → `ORDER BY ... ASC/DESC` verwenden, für LIMIT und OFFSET eine eindeutige Reihenfolge festlegen, bspw. `ORDER BY price DESC, bicycleid`, fehlende Werte bei Bedarf mit NULLS LAST ans Ende stellen
10. Ergebnis fachlich plausibel? → Zeilenanzahl, bekannte Objektanzahlen, Summen und Fälle mit 0 prüfen, ein leeres Ergebnis kann korrekt sein

### Wichtige Ergänzungen
11. Trotz LEFT JOIN fehlen Objekte ohne passende Verkäufe im Zeitraum? → Datumsfilter der rechten Tabelle in ON setzen, wenn alle linken Objekte erhalten bleiben sollen, derselbe Filter in WHERE entfernt die ergänzten NULL-Zeilen
12. Summe fehlt statt 0? → SUM, AVG, MIN und MAX ignorieren NULL und liefern ohne bekannte Werte NULL, falls fachlich passend `COALESCE(SUM(betrag), 0)` verwenden
13. Anzahl oder Umsatz zu hoch? → Mehrere 1:n-JOINs können Zeilen vervielfachen, Beziehungen prüfen und nötigenfalls vor dem JOIN aggregieren, `COUNT(DISTINCT id)` zählt eindeutige Objekte, `SUM(DISTINCT betrag)` ist keine allgemeine Reparatur, weil echte Geschäftsfälle denselben Betrag haben können
14. NOT IN mit Unterabfrage liefert unerwartet keine Treffer? → NULL in der Unterabfrage ausschliessen oder NOT EXISTS verwenden, bei NULL in der linken Vergleichsspalte ebenfalls die gewünschte Behandlung festlegen
15. AND und OR kombiniert? → AND bindet stärker als OR, gewünschte Logik einklammern, bspw. `WHERE (make = 'Bianchi' OR make = 'Trek') AND price > 3000`
16. Datum oder Preis an einer Grenze falsch gefiltert? → BETWEEN schliesst beide Grenzen ein, für ganze Jahre inkl. Uhrzeiten besser `datum >= '2023-01-01' AND datum < '2024-01-01'` verwenden
17. Textmuster verwechselt? → Bei LIKE steht `%` für beliebig viele Zeichen inkl. keinem Zeichen und `_` für genau ein Zeichen, ILIKE ignoriert Gross- und Kleinschreibung, bei Regex berücksichtigt `~` die Schreibweise und `~*` ignoriert sie, `^` steht für Anfang und `$` für Ende
18. Regex-Backslashes oder Prozentzeichen verursachen Probleme? → Python-Strings mit Regex-Backslashes als `r"""..."""` schreiben, im bisherigen Abfragemuster mit psycopg2 Prozentzeichen verdoppeln, bspw. `LIKE 'B%%'`
19. Typfehler bei ROUND oder EXTRACT? → Für `ROUND(..., 2)` bei Gleitkommawerten zuerst in NUMERIC umwandeln, bspw. `ROUND(CAST(AVG(price) AS NUMERIC), 2)`, für EXTRACT muss die Quelle einen geeigneten Datums- oder Zeittyp haben
20. Alias oder Syntax wird nicht erkannt? → SELECT-Aliase in ORDER BY verwenden, in WHERE und HAVING den ursprünglichen Ausdruck schreiben, Kommas zwischen Spalten setzen und keines direkt vor FROM, Textwerte mit `'...'` und Bezeichner bei Bedarf mit `"..."`
21. Tabelle oder Python-Variable fehlt? → Datenbank, Schema, Imports, Verbindungsvariable und Datenimport prüfen, eine Quelldatei im Repo ist noch keine PostgreSQL-Tabelle und jeder Notebook-Kernel benötigt seine eigenen Python-Variablen
22. Falscher Datensatz in einer Tabelle? → Prüfen, welche Daten zuletzt in die Tabelle geschrieben wurden, `if_exists='replace'` ersetzt vorhandene Tabellen, bspw. können Cinema-Kunden die Bike-Shop-Kunden in `customers` ersetzen
23. UPDATE oder DELETE betrifft zu viele Zeilen? → WHERE kontrollieren, ohne Bedingung werden alle Zeilen betroffen, die Auswahl vorher mit SELECT und derselben Bedingung prüfen

### Mehrere LEFT JOINs hintereinander
- Sollen alle Kinos erscheinen, bei `cinemas` beginnen und sowohl `screenings` als auch `tickets` mit LEFT JOIN ergänzen
- Ein nachfolgender INNER JOIN kann zuvor erhaltene Zeilen ohne Treffer wieder entfernen
- Verknüpfungsweg im Mock Exam: `cinemas → screenings → tickets`
- Für die Ticketanzahl `COUNT(t.ticketid)` statt `COUNT(*)` verwenden, damit Kinos ohne Tickets korrekt 0 erhalten

### Keine Treffer oder keine Ergebniszeile?
- COUNT liefert ohne Treffer 0, erzeugt aber keine fehlenden Gruppen
- Wird ein Kino durch einen INNER JOIN ausgeschlossen, kann COUNT es nicht zurückbringen, zuerst den JOIN korrigieren
- Ohne GROUP BY liefert COUNT bei leerer Auswahl eine Ergebniszeile mit 0
- Mit GROUP BY entstehen bei leerer Auswahl keine Gruppen und damit keine Ergebniszeilen

### Änderungen im Mock Exam begründen
- Fehler benennen, welche SQL-Stelle verursacht das falsche Ergebnis?
- Auswirkung erklären, welche Objekte fehlen oder welche Werte werden falsch gezählt?
- Korrektur begründen, warum liefert die geänderte Abfrage das verlangte Ergebnis?
- Jede Änderung einzeln erklären, bspw. Ausgangstabelle, JOIN-Typ und gezählte Spalte

## SQL-Reihenfolge

### Schreibreihenfolge

Nur die benötigten Bestandteile verwenden

```sql
SELECT DISTINCT
    gruppenspalte,
    COUNT(*) AS anzahl,
    SUM(wertspalte) AS summe,
    ROUND(CAST(AVG(wertspalte) AS NUMERIC), 2) AS durchschnitt,
    MIN(wertspalte) AS minimum,
    MAX(wertspalte) AS maximum
FROM tabelle1 AS a
LEFT JOIN tabelle2 AS b
    ON a.id = b.fremdschluessel
WHERE bedingung
GROUP BY gruppenspalte
HAVING COUNT(*) > grenzwert
ORDER BY anzahl DESC, gruppenspalte ASC
LIMIT anzahl_zeilen
OFFSET uebersprungene_zeilen
```

| Bestandteil | Zweck |
|---|---|
| `SELECT` | Ausgabespalten und berechnete Werte bestimmen |
| `DISTINCT` | Doppelte vollständige Ergebniszeilen entfernen |
| `FROM` | Ausgangstabelle bestimmen |
| `JOIN ... ON` | Weitere Tabellen über passende Spalten verbinden |
| `WHERE` | Einzelne Zeilen vor der Aggregation filtern |
| `GROUP BY` | Eine Gruppe pro Wert oder Wertekombination bilden |
| `HAVING` | Gruppen anhand einer Bedingung filtern, bspw. anhand von COUNT oder AVG |
| `ORDER BY` | Ergebnis sortieren, je Spalte mit ASC oder DESC und bei Bedarf NULLS FIRST oder NULLS LAST |
| `LIMIT` | Ausgabe auf maximal die angegebene Zeilenanzahl begrenzen |
| `OFFSET` | Angegebene Anzahl Zeilen überspringen |

### Ausdrücke innerhalb von SELECT
- Einzelne Spalten, `*`, Berechnungen und Ausdrücke mit `AS alias`
- Aggregatfunktionen wie COUNT, SUM, AVG, MIN und MAX
- Funktionen wie ROUND, CAST, EXTRACT, COALESCE und Textfunktionen
- Fallunterscheidungen mit `CASE WHEN ... THEN ... ELSE ... END`
- Fensterfunktionen wie `AVG(...) OVER (...)`, ROW_NUMBER und RANK
- Diese Funktionen haben keine eigene feste Position zwischen den SQL-Klauseln, sie stehen innerhalb der jeweiligen Ausdrücke

### JOINs und Bedingungen
- Statt LEFT JOIN je nach Aufgabe INNER JOIN, RIGHT JOIN, FULL OUTER JOIN oder CROSS JOIN verwenden
- Mehrere JOINs hintereinander sind möglich, CROSS JOIN benötigt keine ON-Bedingung
- WHERE und HAVING können mehrere Bedingungen mit AND, OR und Klammern enthalten
- IN, BETWEEN, LIKE, Regex, EXISTS und Unterabfragen sind Bestandteile von Ausdrücken oder Bedingungen, keine zusätzlichen Schritte der Reihenfolge

### UNION und UNION ALL
- UNION oder UNION ALL steht zwischen vollständigen SELECT-Abfragen
- UNION entfernt doppelte Ergebniszeilen, UNION ALL behält sie
- Beide Teilabfragen benötigen gleich viele Spalten mit kompatiblen Datentypen in derselben Reihenfolge
- Ein abschliessendes ORDER BY, LIMIT oder OFFSET gilt für das gesamte vereinigte Ergebnis
- Soll eine einzelne Teilabfrage separat sortiert oder begrenzt werden, diese einklammern

### Logische Auswertung, vereinfacht
`FROM/JOIN → WHERE → GROUP BY/Aggregation → HAVING → Fensterfunktionen/SELECT → DISTINCT → UNION → ORDER BY → OFFSET/LIMIT`

- Schreibreihenfolge und logische Auswertung sind nicht dasselbe
- OFFSET überspringt zuerst Zeilen, LIMIT begrenzt die danach ausgegebene Anzahl
- SELECT-Aliase sind in ORDER BY verfügbar, in WHERE und HAVING den ursprünglichen Ausdruck verwenden

### Andere SQL-Anweisungen
- INSERT, UPDATE und DELETE verändern Daten
- CREATE, ALTER und DROP erstellen, verändern oder entfernen Datenbankobjekte
- Diese Anweisungen haben eine eigene Syntax und gehören nicht in die SELECT-Reihenfolge

Merksatz: Erst Tabellen verbinden und Zeilen filtern, dann Kennzahlen berechnen und Gruppen filtern, anschliessend Ausgabe bilden, sortieren und begrenzen